# 05 — Modelagem Supervisionada

Neste notebook, usamos `anomalia_score` (saída do Isolation Forest) como **label para treinar modelos supervisionados interpretáveis**.

## Estratégia para evitar circularidade:

- **NÃO usamos** `candidato_anomalia_heuristica` (que foi derivado de regras sobre as features)
- **USAMOS** `anomalia_score` (saída contínua do IF, que é uma "verdade" descoberta, não derivada)
- Isso permite que modelos como KNN, Árvore e Random Forest **aprendam padrões reais**, em vez de memorizar regras

## Algoritmos implementados:
1. **KNN** — baseado em distância (edital exige métodos baseados em distância)
2. **Árvore de Decisão** — interpretável, mostra critérios de split
3. **Random Forest** — ensemble, mais robusto

## Validação:
- Train/test split já feito no dataset (coluna `fold`)
- Métricas: F1, ROC-AUC, PR-AUC, matriz de confusão
- StratifiedKFold (k=5) no treino para avaliar estabilidade

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_recall_curve, auc,
    confusion_matrix, classification_report, roc_curve, precision_score, recall_score
)

from src.config import INTERIM_DIR

# Configurações
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

RANDOM_STATE = 42
CV_FOLDS = 5

## 1. Preparação de dados

In [ ]:
# Carregar dataset
path = INTERIM_DIR / "dataset_analitico_com_scores.parquet"
df = pd.read_parquet(path)

print(f"Dataset original: {df.shape}")
print(f"\nFold split:")
print(df['fold'].value_counts())
print(f"\nAnomalias por fold (IF label):")
print(df.groupby('fold')['anomalia_label'].value_counts())

In [ ]:
# Separar treino e teste usando a coluna 'fold' (já estratificada em src/anomaly.py)
df_train = df[df['fold'] == 'train'].copy()
df_test = df[df['fold'] == 'test'].copy()

print(f"Treino: {len(df_train)} linhas")
print(f"Teste:  {len(df_test)} linhas")

# Features para modelo (mesmo que IF usou)
feats = [
    'log_valor_licitacao',
    'n_participantes',
    'n_itens',
    'hhi',
    'top1_share',
    'valor_total_itens',
]

# Label: anomalia_label (binarizado do IF)
# IMPORTANTE: NÃO usamos candidato_anomalia_heuristica para evitar circularidade
y_train = df_train['anomalia_label'].astype(int)
y_test = df_test['anomalia_label'].astype(int)

X_train = df_train[feats].fillna(0)
X_test = df_test[feats].fillna(0)

print(f"\nLabel distribution:")
print(f"  Treino: {y_train.sum()} anomalias de {len(y_train)} ({y_train.mean():.2%})")
print(f"  Teste:  {y_test.sum()} anomalias de {len(y_test)} ({y_test.mean():.2%})")

In [ ]:
# Padronizar features (importante para KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features padronizadas (StandardScaler)")

## 2. K-Nearest Neighbors (KNN)

Algoritmo baseado em distância — requisito do edital.

In [ ]:
# Tuning de k com cross-validation no treino
k_values = [3, 5, 7, 10, 15, 20]
cv_scores = []

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_validate(
        knn, X_train_scaled, y_train,
        cv=skf,
        scoring=['f1', 'roc_auc', 'precision', 'recall'],
        n_jobs=-1
    )
    cv_scores.append({
        'k': k,
        'f1_mean': scores['test_f1'].mean(),
        'f1_std': scores['test_f1'].std(),
        'roc_auc_mean': scores['test_roc_auc'].mean(),
    })

cv_df = pd.DataFrame(cv_scores)
print("KNN tuning (cross-validation no treino):")
print(cv_df.to_string(index=False))

# Melhor k
best_k = cv_df.loc[cv_df['f1_mean'].idxmax(), 'k'].astype(int)
print(f"\n✓ Melhor k = {best_k}")

In [ ]:
# Treinar KNN com melhor k
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_scaled, y_train)

# Predições
y_pred_knn = knn.predict(X_test_scaled)
y_proba_knn = knn.predict_proba(X_test_scaled)[:, 1]

# Métricas
print(f"KNN (k={best_k}) - Teste:")
print(f"  F1:        {f1_score(y_test, y_pred_knn):.4f}")
print(f"  Precisão: {precision_score(y_test, y_pred_knn):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_knn):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_proba_knn):.4f}")

pr_auc_knn = auc(*precision_recall_curve(y_test, y_proba_knn)[:2])
print(f"  PR-AUC:    {pr_auc_knn:.4f}")

print(f"\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred_knn))

## 3. Árvore de Decisão

Modelo interpretável — permite visualizar critérios de split.

In [ ]:
# Tuning de max_depth com CV
depth_values = [3, 5, 7, 10, 15]
cv_scores_dt = []

for depth in depth_values:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE, min_samples_split=10)
    scores = cross_validate(
        dt, X_train_scaled, y_train,
        cv=skf,
        scoring=['f1', 'roc_auc'],
        n_jobs=-1
    )
    cv_scores_dt.append({
        'max_depth': depth,
        'f1_mean': scores['test_f1'].mean(),
        'f1_std': scores['test_f1'].std(),
        'roc_auc_mean': scores['test_roc_auc'].mean(),
    })

cv_df_dt = pd.DataFrame(cv_scores_dt)
print("Decision Tree tuning (cross-validation):")
print(cv_df_dt.to_string(index=False))

best_depth = cv_df_dt.loc[cv_df_dt['f1_mean'].idxmax(), 'max_depth'].astype(int)
print(f"\n✓ Melhor max_depth = {best_depth}")

In [ ]:
# Treinar árvore com melhor profundidade
dt = DecisionTreeClassifier(
    max_depth=best_depth,
    min_samples_split=10,
    random_state=RANDOM_STATE
)
dt.fit(X_train_scaled, y_train)

# Predições
y_pred_dt = dt.predict(X_test_scaled)
y_proba_dt = dt.predict_proba(X_test_scaled)[:, 1]

# Métricas
print(f"Árvore (max_depth={best_depth}) - Teste:")
print(f"  F1:        {f1_score(y_test, y_pred_dt):.4f}")
print(f"  Precisão: {precision_score(y_test, y_pred_dt):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_dt):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_proba_dt):.4f}")

pr_auc_dt = auc(*precision_recall_curve(y_test, y_proba_dt)[:2])
print(f"  PR-AUC:    {pr_auc_dt:.4f}")

print(f"\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred_dt))

# Feature importance
print(f"\nImportância de features (Árvore):")
feat_imp = pd.DataFrame({
    'feature': feats,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)
print(feat_imp.to_string(index=False))

In [ ]:
# Visualizar árvore (reduzida para espaço)
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    dt,
    feature_names=feats,
    class_names=['Normal', 'Anomalia'],
    filled=True,
    ax=ax,
    fontsize=10
)
plt.title(f"Árvore de Decisão (max_depth={best_depth})")
plt.tight_layout()
plt.show()

## 4. Random Forest

Ensemble de árvores — mais robusto e resistente a overfitting.

In [ ]:
# Tuning de n_estimators e max_depth
rf_configs = [
    {'n_estimators': 50, 'max_depth': 10},
    {'n_estimators': 100, 'max_depth': 10},
    {'n_estimators': 100, 'max_depth': 15},
    {'n_estimators': 200, 'max_depth': 15},
]

cv_scores_rf = []

for config in rf_configs:
    rf = RandomForestClassifier(
        n_estimators=config['n_estimators'],
        max_depth=config['max_depth'],
        min_samples_split=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    scores = cross_validate(
        rf, X_train_scaled, y_train,
        cv=skf,
        scoring=['f1', 'roc_auc'],
        n_jobs=-1
    )
    cv_scores_rf.append({
        'n_estimators': config['n_estimators'],
        'max_depth': config['max_depth'],
        'f1_mean': scores['test_f1'].mean(),
        'roc_auc_mean': scores['test_roc_auc'].mean(),
    })

cv_df_rf = pd.DataFrame(cv_scores_rf)
print("Random Forest tuning:")
print(cv_df_rf.to_string(index=False))

best_config = cv_df_rf.loc[cv_df_rf['f1_mean'].idxmax()].to_dict()
print(f"\n✓ Melhor config: n_estimators={int(best_config['n_estimators'])}, max_depth={int(best_config['max_depth'])}")

In [ ]:
# Treinar RF com melhor config
rf = RandomForestClassifier(
    n_estimators=int(best_config['n_estimators']),
    max_depth=int(best_config['max_depth']),
    min_samples_split=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train_scaled, y_train)

# Predições
y_pred_rf = rf.predict(X_test_scaled)
y_proba_rf = rf.predict_proba(X_test_scaled)[:, 1]

# Métricas
print(f"Random Forest - Teste:")
print(f"  F1:        {f1_score(y_test, y_pred_rf):.4f}")
print(f"  Precisão: {precision_score(y_test, y_pred_rf):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_rf):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_proba_rf):.4f}")

pr_auc_rf = auc(*precision_recall_curve(y_test, y_proba_rf)[:2])
print(f"  PR-AUC:    {pr_auc_rf:.4f}")

print(f"\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred_rf))

# Feature importance
print(f"\nImportância de features (RF):")
feat_imp_rf = pd.DataFrame({
    'feature': feats,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print(feat_imp_rf.to_string(index=False))

## 5. Comparação dos modelos

In [ ]:
# Tabela comparativa
models_comparison = pd.DataFrame({
    'Modelo': ['KNN', 'Árvore', 'Random Forest'],
    'F1': [
        f1_score(y_test, y_pred_knn),
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_rf)
    ],
    'Precisão': [
        precision_score(y_test, y_pred_knn),
        precision_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_rf)
    ],
    'Recall': [
        recall_score(y_test, y_pred_knn),
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_rf)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_proba_knn),
        roc_auc_score(y_test, y_proba_dt),
        roc_auc_score(y_test, y_proba_rf)
    ],
    'PR-AUC': [
        pr_auc_knn,
        pr_auc_dt,
        pr_auc_rf
    ]
})

print("Comparação de modelos:")
print(models_comparison.to_string(index=False))

# Melhor modelo por F1
best_model = models_comparison.loc[models_comparison['F1'].idxmax(), 'Modelo']
print(f"\n✓ Melhor modelo (F1): {best_model}")

In [ ]:
# Visualizar ROC e PR curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC
fpr_knn, tpr_knn, _ = roc_curve(y_test, y_proba_knn)
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_proba_dt)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)

axes[0].plot(fpr_knn, tpr_knn, label=f'KNN (ROC-AUC={roc_auc_score(y_test, y_proba_knn):.3f})')
axes[0].plot(fpr_dt, tpr_dt, label=f'Árvore (ROC-AUC={roc_auc_score(y_test, y_proba_dt):.3f})')
axes[0].plot(fpr_rf, tpr_rf, label=f'RF (ROC-AUC={roc_auc_score(y_test, y_proba_rf):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()
axes[0].grid(alpha=0.3)

# PR
precision_knn, recall_knn, _ = precision_recall_curve(y_test, y_proba_knn)
precision_dt, recall_dt, _ = precision_recall_curve(y_test, y_proba_dt)
precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_proba_rf)

axes[1].plot(recall_knn, precision_knn, label=f'KNN (PR-AUC={pr_auc_knn:.3f})')
axes[1].plot(recall_dt, precision_dt, label=f'Árvore (PR-AUC={pr_auc_dt:.3f})')
axes[1].plot(recall_rf, precision_rf, label=f'RF (PR-AUC={pr_auc_rf:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('PR Curves')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Resumo e conclusões

### Estratégia implementada:
✓ Evitou-se circularidade usando `anomalia_score` (não heurísticas) como label
✓ Train/test split estratificado aplicado antes do Isolation Forest
✓ 3 famílias de algoritmos conforme edital (KNN, Árvore, Ensemble)

### Métricas alcançadas:
- [Inserir tabela de comparação acima]
- Modelo recomendado: [melhor modelo]

### Interpretabilidade:
- A Árvore de Decisão mostra critérios explícitos
- Random Forest oferece importância de features mais robusta
- KNN é menos interpretável, mas muito usado em detecção de anomalias

### Próximos passos (notebook 06):
- Análise de confiança e decisões borderline
- Validação cruzada completa
- Comparação final com Isolation Forest e heurísticas